# Single-agent results — standardised non-Hard benchmark

Hard scenarios are excluded from **all** reported results. This notebook uses the checked exports
from `graphrag_single_agent_comparison.ipynb`; execute that notebook first after any input changes.
Exactly four domains × four scenario types × five models. Original notebooks are preserved in
`backups/before_non_hard_standardization_2026-10-07/`.

With Store accuracy averages v15/v16 equally within each combination; No Store averages v1/v2.
The normalizer maps With Store v15 ↔ No Store v2, and v16 ↔ v1. General metric comparisons pool both versions within each condition. Explicit prompt comparisons and Evidence F1 retain version splits.
The original raw/exploratory notebook views are consolidated here to avoid stale outputs, incorrect
prompt labels, missing setup variables and averaging standard deviations. Noise and Hard charts are
outside this benchmark and are not reported.

Historical Qwen is `hoangquan456/qwen3-nothink:4b`; GraphRAG uses `qwen3:4b` with thinking disabled.
The shared display label does not imply an identical checkpoint or experimental conditions.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from scripts.analysis_reporting import (
    OUTPUT, DOMAINS, MODELS, verify_sources, bar_chart, scenario_name, export_table,
)
metadata = verify_sources()
CHARTS = OUTPUT / "eval_results_analysis"
CHARTS.mkdir(parents=True, exist_ok=True)
accuracy = pd.read_csv(OUTPUT / "accuracy_cells.csv", index_col=0)
accuracy = accuracy[accuracy.System.ne("GraphRAG (v1)")].copy()
metrics = pd.read_csv(OUTPUT / "metric_cells.csv", index_col=0)
metrics = metrics[metrics.System.str.startswith(("With Store", "No Store"))].copy()
coverage = pd.read_csv(OUTPUT / "metric_coverage.csv", index_col=0)
coverage = coverage[coverage.System.str.startswith(("With Store", "No Store"))]
assert set(accuracy.Model) == set(MODELS)
assert set(accuracy.Domain) == set(DOMAINS)
assert not accuracy.Domain.str.endswith("_hard").any()
assert not accuracy.duplicated(["Model", "Domain", "System"]).any()
assert accuracy.groupby(["Model", "System"]).size().eq(16).all()
assert accuracy.Accuracy.notna().all()
display(coverage[coverage.missing.gt(0)])

def plot(table, title, ylabel, ylim=None):
    bar_chart(table, title, ylabel, ylim, folder=CHARTS, colors=("#2b6cb0", "#c05621", "#059669", "#d97706"))

systems = ["With Store (mean v15/v16)", "No Store (mean v1/v2)"]

# General comparisons pool all available summaries from both prompt versions.
prompt_conditions = {
    "With Store (v15)": "With Store (pooled v15/v16)",
    "With Store (v16)": "With Store (pooled v15/v16)",
    "No Store (v1)": "No Store (pooled v1/v2)",
    "No Store (v2)": "No Store (pooled v1/v2)",
}
assert set(metrics.System) == set(prompt_conditions)
pooled_metrics = metrics.assign(System=metrics.System.map(prompt_conditions))
assert len(pooled_metrics) == len(metrics)
assert pooled_metrics.System.notna().all()


,Model,Metric,System,available,expected,missing
11,"Qwen3:4b, no thinking",Average_BCR_BeliefCoverage,No Store (v1),0,16,16
12,"Qwen3:4b, no thinking",Average_BCR_BeliefCoverage,No Store (v2),0,16,16
16,"Qwen3:4b, no thinking",Average_Evidence_Canonical_Keys,No Store (v1),0,16,16
17,"Qwen3:4b, no thinking",Average_Evidence_Canonical_Keys,No Store (v2),0,16,16
21,"Qwen3:4b, no thinking",Average_Evidence_Cited_Keys,No Store (v1),0,16,16
...,...,...,...,...,...,...
536,ministral-3:3b,TPCA_Words_Wrong,No Store (v1),15,16,1
538,ministral-3:3b,TPCA_Words_Wrong,With Store (v15),11,16,5
539,ministral-3:3b,TPCA_Words_Wrong,With Store (v16),10,16,6
546,ministral-3:3b,WCT_WallClock_PerTurn_Seconds,No Store (v1),0,16,16


## Accuracy and scenario coverage
Sample SD describes spread across prompt-averaged model/combination cells, not uncertainty across runs.

In [2]:
for dimension in ["Model", "Scenario", "Family"]:
    summary = accuracy.groupby([dimension, "System"]).Accuracy.agg(mean="mean", sd="std", n="count")
    display(summary)
    export_table(summary, "accuracy_" + dimension.lower(), CHARTS)
    plot(summary["mean"].unstack().reindex(columns=systems), "Accuracy ↑ by " + dimension.lower(), "Accuracy ↑", (0, 1))
overall = accuracy.groupby("System").Accuracy.agg(mean="mean", sd="std", n="count")
display(overall)
plot(overall["mean"].to_frame().T, "Overall accuracy ↑", "Accuracy ↑", (0, 1))


mean        sd   n
Model                 System                                           
Qwen3:4b, no thinking No Store (mean v1/v2)      0.623750  0.212662  16
                      With Store (mean v15/v16)  0.896875  0.131882  16
gemma3:1b             No Store (mean v1/v2)      0.369687  0.125359  16
                      With Store (mean v15/v16)  0.561250  0.281084  16
gemma4:e2b            No Store (mean v1/v2)      0.552812  0.189578  16
                      With Store (mean v15/v16)  0.839688  0.187772  16
llama3.2:1b           No Store (mean v1/v2)      0.174687  0.143927  16
                      With Store (mean v15/v16)  0.388437  0.219816  16
ministral-3:3b        No Store (mean v1/v2)      0.601875  0.185273  16
                      With Store (mean v15/v16)  0.885000  0.137635  16

/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


mean        sd   n
Scenario           System                                          
absurd             No Store (mean v1/v2)      0.44975  0.219278  20
                   With Store (mean v15/v16)  0.81075  0.213753  20
absurd_temporal    No Store (mean v1/v2)      0.39675  0.189739  20
                   With Store (mean v15/v16)  0.73300  0.224678  20
belief_maintenance No Store (mean v1/v2)      0.60750  0.244085  20
                   With Store (mean v15/v16)  0.85525  0.183041  20
grounding          No Store (mean v1/v2)      0.40425  0.260739  20
                   With Store (mean v15/v16)  0.45800  0.319598  20

/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


mean        sd   n
Family       System                                          
alien_clinic No Store (mean v1/v2)      0.39400  0.191597  20
             With Store (mean v15/v16)  0.67150  0.263749  20
crime_scene  No Store (mean v1/v2)      0.47200  0.244198  20
             With Store (mean v15/v16)  0.72775  0.297951  20
loan         No Store (mean v1/v2)      0.42925  0.268231  20
             With Store (mean v15/v16)  0.70700  0.301476  20
thorncrester No Store (mean v1/v2)      0.56300  0.238644  20
             With Store (mean v15/v16)  0.75075  0.281492  20

/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


,mean,sd,n
System,,,
No Store (mean v1/v2),0.464562,0.241278,80
With Store (mean v15/v16),0.714250,0.282590,80


/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Prompt versions
Same 16 combinations per model and prompt; no mixing of conditions under a prompt-only label.

In [3]:
variants = pd.read_csv(OUTPUT / "accuracy_prompt_cells.csv")
variant_columns = ["With Store (v15)", "With Store (v16)", "No Store (v1)", "No Store (v2)"]
version_means = variants.groupby("Model")[variant_columns].mean()
display(version_means)
plot(version_means, "Accuracy ↑ by explicit prompt version", "Accuracy ↑", (0, 1))


,With Store (v15),With Store (v16),No Store (v1),No Store (v2)
Model,,,,
"Qwen3:4b, no thinking",0.935625,0.858125,0.741250,0.506250
gemma3:1b,0.677500,0.445000,0.320625,0.418750
gemma4:e2b,0.860000,0.819375,0.756875,0.348750
llama3.2:1b,0.470625,0.306250,0.058750,0.290625
ministral-3:3b,0.909375,0.860625,0.577500,0.626250


/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Calibration, extraction and stability

With Store pools both evaluation prompt versions; No Store pools both baseline prompt versions,
as in the original notebook. Each available model/domain/prompt summary contributes equally.
The table reports mean, sample SD, and number of available summaries (up to 32 per model and
condition: 16 combinations × 2 versions). Missing values are omitted, never filled with zero;
counts may differ across conditions. These are averages of stored calibration summaries,
not calibration recomputed from pooled individual predictions.


In [4]:
labels = metadata["metric_definitions"]
for metric in ["Summary_Metric_Brier_Score", "Summary_Metric_ECE", "Summary_Metric_Log_Loss",
               "Summary_Metric_MacroCE", "AFR_Answer_Flip_Rate", "DS_Determinism_Score", "EFR_Extraction_Failure_Rate"]:
    subset = pooled_metrics[pooled_metrics.Metric.eq(metric)]
    stats = subset.groupby(["Model", "System"]).Value.agg(mean="mean", sd="std", n="count")
    display(stats)
    export_table(stats, metric + "_summary", CHARTS)
    plot(stats["mean"].unstack(), f"{labels[metric]} · both versions pooled, available summaries", labels[metric])


mean        sd   n
Model                 System                                             
Qwen3:4b, no thinking No Store (pooled v1/v2)      0.336657  0.231836  32
                      With Store (pooled v15/v16)  0.098081  0.141103  32
gemma3:1b             No Store (pooled v1/v2)      0.540078  0.149460  32
                      With Store (pooled v15/v16)  0.357076  0.257607  32
gemma4:e2b            No Store (pooled v1/v2)      0.279458  0.209694  32
                      With Store (pooled v15/v16)  0.141758  0.201526  32
llama3.2:1b           No Store (pooled v1/v2)      0.361656  0.166203  28
                      With Store (pooled v15/v16)  0.325827  0.164937  32
ministral-3:3b        No Store (pooled v1/v2)      0.253826  0.135075  32
                      With Store (pooled v15/v16)  0.099954  0.115957  32

/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


mean        sd   n
Model                 System                                             
Qwen3:4b, no thinking No Store (pooled v1/v2)      0.334283  0.239156  32
                      With Store (pooled v15/v16)  0.098479  0.141229  32
gemma3:1b             No Store (pooled v1/v2)      0.558614  0.154449  32
                      With Store (pooled v15/v16)  0.368388  0.278051  32
gemma4:e2b            No Store (pooled v1/v2)      0.276798  0.210727  32
                      With Store (pooled v15/v16)  0.143754  0.204557  32
llama3.2:1b           No Store (pooled v1/v2)      0.418526  0.207025  28
                      With Store (pooled v15/v16)  0.340088  0.193698  32
ministral-3:3b        No Store (pooled v1/v2)      0.241304  0.152456  32
                      With Store (pooled v15/v16)  0.103692  0.111870  32

/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


mean        sd   n
Model                 System                                             
Qwen3:4b, no thinking No Store (pooled v1/v2)      3.352311  2.568825  32
                      With Store (pooled v15/v16)  1.231364  1.799813  32
gemma3:1b             No Store (pooled v1/v2)      3.381890  1.236825  32
                      With Store (pooled v15/v16)  2.024976  1.465391  32
gemma4:e2b            No Store (pooled v1/v2)      2.987778  2.599506  32
                      With Store (pooled v15/v16)  1.820573  2.753601  32
llama3.2:1b           No Store (pooled v1/v2)      1.155545  0.533669  28
                      With Store (pooled v15/v16)  1.287381  0.743672  32
ministral-3:3b        No Store (pooled v1/v2)      1.055376  0.728052  32
                      With Store (pooled v15/v16)  0.514522  0.794746  32

/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


... (4 outputs left out)

mean        sd   n
Model                 System                                             
Qwen3:4b, no thinking No Store (pooled v1/v2)      0.768750  0.161520  32
                      With Store (pooled v15/v16)  0.959375  0.066524  32
gemma3:1b             No Store (pooled v1/v2)      0.621875  0.201181  32
                      With Store (pooled v15/v16)  0.781250  0.167404  32
gemma4:e2b            No Store (pooled v1/v2)      0.653125  0.198355  32
                      With Store (pooled v15/v16)  0.931250  0.117604  32
llama3.2:1b           No Store (pooled v1/v2)      0.453125  0.368943  32
                      With Store (pooled v15/v16)  0.709375  0.146704  32
ministral-3:3b        No Store (pooled v1/v2)      0.681250  0.176777  32
                      With Store (pooled v15/v16)  0.921875  0.103906  32

/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


mean        sd   n
Model                 System                                             
Qwen3:4b, no thinking No Store (pooled v1/v2)      0.027187  0.064968  32
                      With Store (pooled v15/v16)  0.016250  0.042331  32
gemma3:1b             No Store (pooled v1/v2)      0.122812  0.131302  32
                      With Store (pooled v15/v16)  0.198125  0.172579  32
gemma4:e2b            No Store (pooled v1/v2)      0.263125  0.291054  32
                      With Store (pooled v15/v16)  0.045000  0.110044  32
llama3.2:1b           No Store (pooled v1/v2)      0.480312  0.423347  32
                      With Store (pooled v15/v16)  0.165000  0.169496  32
ministral-3:3b        No Store (pooled v1/v2)      0.118750  0.207532  32
                      With Store (pooled v15/v16)  0.004375  0.018826  32

/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Evidence, fidelity and response length

Evidence F1 retains the explicit prompt-version comparison from the original notebook.
All other comparisons here pool both prompt versions within each condition, using available
summaries and showing their counts. Evidence/fidelity belongs to With Store only; timing is
parallel-batch time, not request latency. TPCA measures words, not tokens. Missing values are
omitted; empty correct/incorrect response groups remain undefined.


In [5]:
for metric in ["Average_Evidence_F1", "Average_Evidence_Precision", "Average_Evidence_Recall",
               "Average_BCR_BeliefCoverage", "Average_SBIR_SpuriousInjection",
               "TPCA_Words_Correct", "TPCA_Words_Wrong", "WCT_WallClock_PerTurn_Seconds"]:
    source = metrics if metric == "Average_Evidence_F1" else pooled_metrics
    subset = source[source.Metric.eq(metric)]
    stats = subset.groupby(["Model", "System"]).Value.agg(mean="mean", sd="std", n="count")
    display(Markdown("**" + metric + " (per-system available support)**"))
    display(stats)
    export_table(stats, metric + "_available_support", CHARTS)
    table = stats["mean"].unstack().dropna(axis=1, how="all")
    label = metric + (" ↑" if metric.startswith("Average_Evidence") or "BCR" in metric else
                      " ↓" if "SBIR" in metric else " (descriptive)")
    if not table.empty:
        plot(table, label + " · available cells, see counts", label)


**Average_Evidence_F1 (per-system available support)**

mean        sd   n
Model                 System                                  
Qwen3:4b, no thinking No Store (v1)          NaN       NaN   0
                      No Store (v2)          NaN       NaN   0
                      With Store (v15)  0.504012  0.167463  16
                      With Store (v16)  0.379663  0.231978  16
gemma3:1b             No Store (v1)          NaN       NaN   0
                      No Store (v2)          NaN       NaN   0
                      With Store (v15)  0.461512  0.147541  16
                      With Store (v16)  0.000000  0.000000  16
gemma4:e2b            No Store (v1)          NaN       NaN   0
                      No Store (v2)          NaN       NaN   0
                      With Store (v15)  0.034619  0.138475  16
                      With Store (v16)  0.594487  0.158373  16
llama3.2:1b           No Store (v1)          NaN       NaN   0
                      No Store (v2)          NaN       NaN   0
                      With Store (v15)  0.200881  0.073499  16
                      With Store (v16)  0.000000  0.000000  16
ministral-3:3b        No Store (v1)          NaN       NaN   0
                      No Store (v2)          NaN       NaN   0
                      With Store (v15)  0.581512  0.119924  16
                      With Store (v16)  0.131187  0.207735  16

/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


**Average_Evidence_Precision (per-system available support)**

mean        sd   n
Model                 System                                             
Qwen3:4b, no thinking No Store (pooled v1/v2)           NaN       NaN   0
                      With Store (pooled v15/v16)  0.510112  0.204483  32
gemma3:1b             No Store (pooled v1/v2)           NaN       NaN   0
                      With Store (pooled v15/v16)  0.291922  0.307881  32
gemma4:e2b            No Store (pooled v1/v2)           NaN       NaN   0
                      With Store (pooled v15/v16)  0.361812  0.348464  32
llama3.2:1b           No Store (pooled v1/v2)           NaN       NaN   0
                      With Store (pooled v15/v16)  0.161100  0.196211  32
ministral-3:3b        No Store (pooled v1/v2)           NaN       NaN   0
                      With Store (pooled v15/v16)  0.392763  0.294546  32

/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


... (14 outputs left out)

/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


**WCT_WallClock_PerTurn_Seconds (per-system available support)**

mean         sd   n
Model                 System                                               
Qwen3:4b, no thinking No Store (pooled v1/v2)            NaN        NaN   0
                      With Store (pooled v15/v16)  13.980562   6.639070  32
gemma3:1b             No Store (pooled v1/v2)            NaN        NaN   0
                      With Store (pooled v15/v16)  11.720187   3.993505  32
gemma4:e2b            No Store (pooled v1/v2)            NaN        NaN   0
                      With Store (pooled v15/v16)  26.155759  11.417643  32
llama3.2:1b           No Store (pooled v1/v2)            NaN        NaN   0
                      With Store (pooled v15/v16)  13.812213   9.166295  32
ministral-3:3b        No Store (pooled v1/v2)            NaN        NaN   0
                      With Store (pooled v15/v16)  17.202950   6.539747  32

/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Paired combination differences and variability
Differences use prompt-averaged accuracy. SD is recomputed from these combination scores with ddof=1. Historical per-run accuracy and turn counts are unavailable; pooled run SD cannot be recovered. Individual raw-score SDs are preserved without averaging.

In [6]:
paired = accuracy.pivot(index=["Model", "Domain"], columns="System", values="Accuracy")
paired["With Store − No Store (pp)"] = 100 * (paired[systems[0]] - paired[systems[1]])
display(paired)
export_table(paired, "accuracy_deltas", CHARTS)
display(paired.groupby(level="Model")["With Store − No Store (pp)"].agg(["mean", "median", "min", "max"]))
spread = accuracy.groupby(["Model", "Scenario", "System"]).Accuracy.agg(mean="mean", sd="std", n="count")
display(spread)
export_table(spread, "accuracy_model_scenario_spread", CHARTS)
raw_spread = pd.read_csv(OUTPUT / "raw_score_variability_cells.csv", index_col=0)
display(raw_spread[raw_spread.System.str.startswith(("With Store", "No Store"))])


System                                                 No Store (mean v1/v2)  \
Model                 Domain                                                   
Qwen3:4b, no thinking alien_clinic_absurd                              0.435   
                      alien_clinic_absurd_temporal                     0.285   
                      alien_clinic_belief_maintenance                  0.750   
                      alien_clinic_grounding                           0.830   
                      crime_scene_absurd                               0.435   
...                                                                      ...   
ministral-3:3b        loan_grounding                                   0.535   
                      thorncrester_absurd                              0.705   
                      thorncrester_absurd_temporal                     0.700   
                      thorncrester_belief_maintenance                  0.840   
                      thorncrester_grounding                           0.695   

System                                                 With Store (mean v15/v16)  \
Model                 Domain                                                       
Qwen3:4b, no thinking alien_clinic_absurd                                  0.970   
                      alien_clinic_absurd_temporal                         0.600   
                      alien_clinic_belief_maintenance                      0.995   
                      alien_clinic_grounding                               0.770   
                      crime_scene_absurd                                   0.975   
...                                                                          ...   
ministral-3:3b        loan_grounding                                       0.690   
                      thorncrester_absurd                                  0.965   
                      thorncrester_absurd_temporal                         0.900   
                      thorncrester_belief_maintenance                      1.000   
                      thorncrester_grounding                               0.780   

System                                                 With Store − No Store (pp)  
Model                 Domain                                                       
Qwen3:4b, no thinking alien_clinic_absurd                                    53.5  
                      alien_clinic_absurd_temporal                           31.5  
                      alien_clinic_belief_maintenance                        24.5  
                      alien_clinic_grounding                                 -6.0  
                      crime_scene_absurd                                     54.0  
...                                                                           ...  
ministral-3:3b        loan_grounding                                         15.5  
                      thorncrester_absurd                                    26.0  
                      thorncrester_absurd_temporal                           20.0  
                      thorncrester_belief_maintenance                        16.0  
                      thorncrester_grounding                                  8.5  

[80 rows x 3 columns]

,mean,median,min,max
Model,,,,
"Qwen3:4b, no thinking",27.31250,27.00,-6.0,62.5
gemma3:1b,19.15625,25.00,-31.0,50.0
gemma4:e2b,28.68750,31.50,-41.0,57.0
llama3.2:1b,21.37500,23.00,-2.0,48.0
ministral-3:3b,28.31250,25.75,3.0,58.0


mean  \
Model                 Scenario           System                               
Qwen3:4b, no thinking absurd             No Store (mean v1/v2)      0.44125   
                                         With Store (mean v15/v16)  0.98250   
                      absurd_temporal    No Store (mean v1/v2)      0.45625   
                                         With Store (mean v15/v16)  0.87125   
                      belief_maintenance No Store (mean v1/v2)      0.88125   
                                         With Store (mean v15/v16)  0.97875   
                      grounding          No Store (mean v1/v2)      0.71625   
                                         With Store (mean v15/v16)  0.75500   
gemma3:1b             absurd             No Store (mean v1/v2)      0.44625   
                                         With Store (mean v15/v16)  0.73000   
                      absurd_temporal    No Store (mean v1/v2)      0.38375   
                                         With Store (mean v15/v16)  0.64625   
                      belief_maintenance No Store (mean v1/v2)      0.37875   
                                         With Store (mean v15/v16)  0.74875   
                      grounding          No Store (mean v1/v2)      0.27000   
                                         With Store (mean v15/v16)  0.12000   
gemma4:e2b            absurd             No Store (mean v1/v2)      0.64875   
                                         With Store (mean v15/v16)  0.88375   
                      absurd_temporal    No Store (mean v1/v2)      0.50500   
                                         With Store (mean v15/v16)  0.85625   
                      belief_maintenance No Store (mean v1/v2)      0.65000   
                                         With Store (mean v15/v16)  0.99000   
                      grounding          No Store (mean v1/v2)      0.40750   
                                         With Store (mean v15/v16)  0.62875   
llama3.2:1b           absurd             No Store (mean v1/v2)      0.17000   
                                         With Store (mean v15/v16)  0.49000   
                      absurd_temporal    No Store (mean v1/v2)      0.14500   
                                         With Store (mean v15/v16)  0.43750   
                      belief_maintenance No Store (mean v1/v2)      0.35250   
                                         With Store (mean v15/v16)  0.56750   
                      grounding          No Store (mean v1/v2)      0.03125   
                                         With Store (mean v15/v16)  0.05875   
ministral-3:3b        absurd             No Store (mean v1/v2)      0.54250   
                                         With Store (mean v15/v16)  0.96750   
                      absurd_temporal    No Store (mean v1/v2)      0.49375   
                                         With Store (mean v15/v16)  0.85375   
                      belief_maintenance No Store (mean v1/v2)      0.77500   
                                         With Store (mean v15/v16)  0.99125   
                      grounding          No Store (mean v1/v2)      0.59625   
                                         With Store (mean v15/v16)  0.72750   

                                                                          sd  \
Model                 Scenario           System                                
Qwen3:4b, no thinking absurd             No Store (mean v1/v2)      0.059634   
                                         With Store (mean v15/v16)  0.013229   
                      absurd_temporal    No Store (mean v1/v2)      0.149464   
                                         With Store (mean v15/v16)  0.186117   
                      belief_maintenance No Store (mean v1/v2)      0.087595   
                                         With Store (mean v15/v16)  0.032755   
                      grounding          No Store (mean v1/v2)      0.104513   
                                

,Model,Domain,Temp,Runs,Metric,Value,System
1122,gemma3:1b,loan_absurd,0,10,Variance_Raw_Score,0.2778,With Store (v15)
1123,gemma3:1b,loan_absurd,0,10,Variance_Raw_Score,0.7667,No Store (v2)
1124,gemma3:1b,loan_absurd,0,10,StdDev_Raw_Score,0.5270,With Store (v15)
1125,gemma3:1b,loan_absurd,0,10,StdDev_Raw_Score,0.8756,No Store (v2)
1158,gemma3:1b,alien_clinic_absurd,0,10,Variance_Raw_Score,0.7667,With Store (v15)
...,...,...,...,...,...,...,...
6791,gemma4:e2b,crime_scene_belief_maintenance,0,10,StdDev_Raw_Score,0.7379,No Store (v1)
6824,gemma4:e2b,thorncrester_belief_maintenance,0,10,Variance_Raw_Score,0.0000,With Store (v16)
6825,gemma4:e2b,thorncrester_belief_maintenance,0,10,Variance_Raw_Score,0.0000,No Store (v1)
6826,gemma4:e2b,thorncrester_belief_maintenance,0,10,StdDev_Raw_Score,0.0000,With Store (v16)


## Checks and findings

In [7]:
for system, columns in [(systems[0], ["With Store (v15)", "With Store (v16)"]),
                         (systems[1], ["No Store (v1)", "No Store (v2)"])]:
    direct = variants.set_index(["Model", "Domain"])[columns].mean(axis=1).sort_index()
    actual = paired[system].sort_index()
    np.testing.assert_allclose(direct, actual)
for frame in [accuracy, metrics]:
    assert not frame.Domain.str.endswith("_hard").any()
    assert set(frame.Domain) == set(DOMAINS)
print("PASS: source hashes, exclusion, 5 × 16 coverage, uniqueness, prompt weights and chart/export agreement.")
display(pd.read_csv(OUTPUT / "before_after_accuracy.csv", index_col=0))
display(Markdown("All five models have higher mean accuracy with the store on this benchmark. "
                 "This is descriptive: matching scenarios does not establish identical questions, histories, seeds, "
                 "execution settings or Qwen checkpoints. See definitions.json for all limitations."))

# Verify pooling includes each version's values and support without selecting one version.
version_stats = metrics.groupby(["Model", "Metric", "System"]).Value.agg(total="sum", n="count").reset_index()
version_stats["System"] = version_stats.System.map(prompt_conditions)
expected = version_stats.groupby(["Model", "Metric", "System"])[["total", "n"]].sum()
actual = pooled_metrics.groupby(["Model", "Metric", "System"]).Value.agg(mean="mean", n="count")
np.testing.assert_allclose(actual["mean"], expected.total.div(expected.n.where(expected.n.ne(0))), equal_nan=True)
np.testing.assert_array_equal(actual.n, expected.n)
print("PASS: general metrics include both prompt versions; Evidence F1 retains its explicit version split.")


PASS: source hashes, exclusion, 5 × 16 coverage, uniqueness, prompt weights and chart/export agreement.


,Model,System,before_combinations,before_accuracy,after_combinations,after_accuracy,before_scope
0,gemma3:1b,With Store (mean v15/v16),16,0.561250,16,0.561250,"all available historical scenarios, including ..."
1,gemma3:1b,No Store (mean v1/v2),16,0.369687,16,0.369688,"all available historical scenarios, including ..."
2,gemma3:1b,GraphRAG (v1),16,0.270625,16,0.270625,GraphRAG already used only the 16 benchmark co...
3,gemma4:e2b,With Store (mean v15/v16),20,0.778750,16,0.839688,"all available historical scenarios, including ..."
4,gemma4:e2b,No Store (mean v1/v2),20,0.509000,16,0.552813,"all available historical scenarios, including ..."
5,gemma4:e2b,GraphRAG (v1),16,0.754375,16,0.754375,GraphRAG already used only the 16 benchmark co...
6,llama3.2:1b,With Store (mean v15/v16),16,0.388437,16,0.388437,"all available historical scenarios, including ..."
7,llama3.2:1b,No Store (mean v1/v2),16,0.174687,16,0.174687,"all available historical scenarios, including ..."
8,llama3.2:1b,GraphRAG (v1),16,0.263750,16,0.263750,GraphRAG already used only the 16 benchmark co...
9,ministral-3:3b,With Store (mean v15/v16),20,0.817500,16,0.885000,"all available historical scenarios, including ..."


All five models have higher mean accuracy with the store on this benchmark. This is descriptive: matching scenarios does not establish identical questions, histories, seeds, execution settings or Qwen checkpoints. See definitions.json for all limitations.

PASS: general metrics include both prompt versions; Evidence F1 retains its explicit version split.
